<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1 prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

## Step 1.1: Configure the environment

In [1]:
from datetime import datetime
from shapely.geometry import Polygon
from shapely import box
from pyproj import CRS 
import folium
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import xarray as xr
import rioxarray
import joblib
from ipyleaflet import basemaps
import pystac_client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import planetary_computer
from odc.stac import load
from pystac.client import Client
from skimage.feature import graycomatrix, graycoprops
import matplotlib.pyplot as plt
from matplotlib import colors
from utils import load_data, load_s1_dem, scale, calculate_band_indices, apply_mask, mask_water, all_masks, do_prediction
from matplotlib import colors

In [2]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

In [3]:
# Predefined variable for title and version

# Enter your initials
initials = "mw"

# Enter your site name
site = "fiji"

# Date
date = datetime.now()

# Make a clean version string
version = f"{initials}-{site}-{date.strftime('%d%m%Y')}"
print(version)

mw-fiji-05052026


## Step 1.2: Configure STAC access and search parameters

In this block we are accessing the digital earth pacific catalog. This catalog housed all the earth observation dataset including
sentinel imagery (S2 GEOMAD) product in this case. Then we will use the Client.open function to open the catalog so that we can 
access it.
>catalog = "https://stac.digitalearthpacific.org"

>client = Client.open(catalog)

<font color='blue'>1.1. Reading the datapoints using the "read_file" function

Your code goes in the cells below. Add more cells here by clicking on the + button above.

First let's try and read your geojson datapoints in using the geopandas (gpd) library and the read_file function   

> training = gpd.read_file("training-data/ filename.geojson")

Second, we have to set the coordinate reference system to a suitable crs (e.g EPSG:4326)

>training = training.to_crs("EPSG:4326")

Define your bounding box by using the .total_bounds function to extract the lower_left furthest point coordinates and
the top right furthest point coordinates and convert then to a bounding box.
> min_lon, min_lat, max_lon, max_lat = training.total_bounds

Assign a new variable called bbox
>bbox = [min_lon, min_lat, max_lon, max_lat]

print the attributes in the dataset
> training

In this code you will search for dataset in the dep catalog using the client.search function and then define the 
collection or product you want to search, then redefine your datetime and bounding box. 

> datetime = "2023"
 
> items = client.search(
    collections=["dep_s2_geomad"],
    datetime=datetime,
    bbox=bbox
).item_collection()

Print the collection
>print(f"Found {len(items)} items in for {datetime}")

Define all the bands in the satellite imagery you have accessed earlier. Load all the bands, the bounding box that define your area of interes
and the location of the collection (imagery)
###### measurements = ["nir", "red", "blue", "green", "emad", "smad", "bcmad", "green", "nir08", "nir09", "swir16", "swir22", "coastal", "rededge1", "rededge2", "rededge3"]

> measurements = ["", "", ""]

> data = load_data(
    items,
    measurements,
    bbox,


> print(data)

This cell initializes a Dask client for parallel processing. The notebook allocates one worker with multiple threads and defines a memory limit to improve performance when working with large geospatial datasets. 

>dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='16GB')

This cell also runs configure_s3_access() to establish secure access to cloud-hosted datasets stored in S3-compatible object storage.

>configure_s3_access(cloud_defaults=True, requester_pays=True)

This cell applies the scale() function to the loaded satellite dataset. Scaling converts raw pixel values into meaningful reflectance values that can be used for scientific analysis. 
>scaled = scale(data)

The .squeeze() method is then applied to remove unnecessary dimensions from the dataset and simplify its structure.
>scaled = scaled.squeeze() # .compute()

This cell calculates additional spectral indices from the satellite imagery using the calculate_band_indices() function. Spectral indices are mathematical combinations of different spectral bands that highlight environmental properties such as vegetation health or water presence. 
>scaled = calculate_band_indices(scaled)

This cell applies the all_masks() function to the dataset. The masking process removes unwanted pixels such as clouds, cloud shadows, water, or invalid observations. The function also returns the generated mask itself so the excluded areas can be inspected if necessary.
>masked_scaled, mask = all_masks(scaled, return_mask = True)

This line converts the masked_scaled dataset into an xarray DataArray using the to_dataarray() function. 
>combined_da = masked_scaled.to_dataarray()

This line removes unnecessary singleton dimensions from the DataArray using the squeeze() function.
>combined_da = combined_da.squeeze()#.stack(dims=["y", "x"])#.transpose()

This line reshapes the two-dimensional spatial coordinates (y and x) into a single dimension called new_dim. The stack() function combines all pixel locations into one continuous dimension, transforming the raster dataset into a structure suitable for machine learning input
>stacked_arrays_2d = combined_da.stack(new_dim=("y", "x"))

This line reorganises the dimensions of the stacked DataArray using the transpose() function. The new dimension order places pixel locations (new_dim) first and spectral variables (variable) second.
>reordered_data_array = stacked_arrays_2d.transpose('new_dim', 'variable')

This line displays the shape of the stacked DataArray.
>stacked_arrays_2d.shape

This line searches the stacked raster dataset for positive infinity values (inf). The where() function keeps only values that are not equal to positive infinity, while infinite values are replaced with NaN (Not a Number).
>'stacked_arrays_2d = stacked_arrays_2d.where(stacked_arrays_2d != float("inf"))'

This line performs the same cleaning process for negative infinity values (-inf). Any negative infinite values found in the dataset are replaced with NaN to ensure the dataset contains only valid numerical values.
>'stacked_arrays_2d = stacked_arrays_2d.where(stacked_arrays_2d != float("-inf"))'

This line replaces any NaN values with 0. The squeeze() function removes unnecessary dimensions from the dataset. The fillna(0) function replaces all missing values with zeros. The transpose() function reorganizes the dimensions so that observations are arranged in rows and variables in columns. Finally, the to_pandas() function converts the xarray object into a Pandas DataFrame, making the dataset easier to manipulate and compatible with Scikit-learn prediction workflows.
>'df = stacked_arrays_2d.squeeze().fillna(0).transpose().to_pandas()'

This line creates a boolean mask that identifies rows where all values are equal to zero.
##### Remove the all-zero rows
> zero_mask = (df == 0).all(axis=1)  # Creates a boolean Series

This line removes all rows identified as entirely zero values.
>non_zero_df = df.loc[~zero_mask]  # Filters out all-zero rows

This line creates a new Pandas Series filled with NaN values. The Series uses the same index as the original DataFrame so that prediction results can later be inserted back into their correct spatial positions.
##### Create a new array to hold the predictions
>full_pred = pd.Series(np.nan, index=df.index)

This line reorganises the dimensions of the stacked raster data using the transpose() function.
>reordered_data_array = stacked_arrays_2d.transpose('new_dim', 'variable')

### Model training

print the dataset attributes
>reordered_data_array

In [19]:
# # Reproject training data to the GeoMAD CRS and convert to xarray
# training_reprojected = training.to_crs(masked_scaled.odc.crs)
# training_da = training_reprojected.assign(
#     x=training_reprojected.geometry.x, y=training_reprojected.geometry.y
# ).to_xarray()

# # Extract training values from the masked dataset
# training_values = (
#     masked_scaled.sel(training_da[["x", "y"]], method="nearest")
#     .squeeze()
#     .compute()
#     .to_pandas()
# )
# training_values

This line loads a previously trained machine learning model from the models directory using the joblib.load() function.
>model = joblib.load("models/filename.model")

This line uses the trained Random Forest machine learning model to generate predictions for the processed raster dataset. The model.predict() function analyses the spectral variables stored in reordered_data_array and predicts a land cover class for every pixel observation.
##### Predict the classes
>predicted = model.predict(reordered_data_array)

This line prints the Coordinate Reference System (CRS) of the raster dataset stored in the variable data. The .rio.crs function from the rioxarray library retrieves the spatial reference information associated with the dataset.
##### print("Prediction Input CRS:", reordered_data_array.rio.crs)
>print("Prediction Input CRS:", data.rio.crs)

This line reshapes the one-dimensional prediction array back into a two-dimensional grid using the reshape() function. The dimensions are defined using the lengths of the y and x coordinates from the original masked_scaled raster dataset.
##### Reshape back to the original 2D array
>reordered_data_array = predicted.reshape(len(masked_scaled.y), len(masked_scaled.x))

This line creates a new xarray DataArray called predicted_da using the reshaped prediction results. The variable reordered_data_array contains the predicted land cover classes arranged in their original raster grid format.
>predicted_da = xr.DataArray(
>>reordered_data_array, coords={"y": masked_scaled.y, "x": masked_scaled.x}, dims=["y", "x"]

>)

The coords parameter assigns the original spatial coordinates from the masked_scaled dataset, ensuring that every prediction is linked to the correct geographic location.

This line prints the data type (dtype) of the predicted raster stored in predicted_da
>print(predicted_da.dtype)  # Check the dtype of your DataArray

This line converts the predicted raster values into the float32 data type using the astype() function.
>predicted_da = predicted_da.astype('float32')  # Convert to float32

This cell checks whether the predicted raster dataset contains any missing values (NaN) and replaces them with a predefined NoData value    
##### Check for NaN values
>if np.isnan(predicted_da).any():
>>  print("NaN values found in the data")
>>> predicted_da = predicted_da.fillna(-9999)

This line converts the masked_scaled dataset from a lazy-loaded Dask object into a fully computed dataset stored in memory.
>masked_scaled = masked_scaled.compute()

This line uses the custom do_prediction() function to apply the trained machine learning model to the processed satellite dataset stored in masked_scaled.
>predicted = do_prediction(masked_scaled, model)

This line displays the contents of the predicted variable in the notebook output.
>predicted

This line applies the spatial mask stored in the variable mask to the predicted raster dataset. The where() function keeps only the pixels where the mask condition is valid and removes or hides pixels outside the masked region.
>predicted = predicted.where(mask)

This line defines a NoData value and stores it in the variable nodata_value. The value -9998 is used as a placeholder to represent missing, masked, or invalid pixels within the raster dataset.
>nodata_value = -9998 # or -9999

This line assigns the NoData value -9998 to the mask layer within the predicted dataset. 
>predicted['mask'] = nodata_value

    This ensures that masked or invalid regions are explicitly labelled with the predefined NoData value rather than remaining empty or undefined.

This line replaces all missing (NaN) values in the predicted raster dataset with -9999 using the fillna() function.
>predicted_filled = predicted.fillna(-9999)

This line converts the prediction raster into integer format in two stages. First, the raster values are converted into floating-point format using astype(float). The values are then converted into integers using astype(int).
>predicted_int = predicted_filled.astype(float).astype(int)

This cell creates a classification legend for the land cover map by defining all land cover classes, their numerical class codes, and the colours that will be used to display them during visualisation.

In [ ]:
# from matplotlib import colors

# classes = [
#     [0, 'No_Data', '#00000000'], # <-- NEW: Code 0 mapped to a light gray color
#     [1, 'classname1', '#064a00'],
#     [2, 'classname2', '#7BB369' ],
#     [3, 'classname3', '#FFEE8C'],
#     [4, 'classname4', '#bd0007'],
#     [5, 'classname5','#919191'],
#     [6, 'classname6','#484E45'],
# ]                            

This line extracts all class codes from the classes list using list comprehension. Each sublist inside classes contains a class code, class name, and colour value. The expression c[0] selects the first element from each sublist, which represents the numerical class code. The extracted values are stored in the variable values_list.
>values_list = [c[0] for c in classes]

This line extracts all hexadecimal colour values from the classes list. The expression c[2] selects the third element from each sublist, which represents the colour assigned to each land cover category. The extracted colours are stored in the variable color_list.
>color_list = [c[2] for c in classes]

This line creates a custom colour map using the ListedColormap() function from Matplotlib. The function uses the colours stored in color_list and assigns them to the land cover classes in the order they were defined.
>c_map = colors.ListedColormap(color_list)

This line creates the class boundaries used for raster visualization. These boundaries help Matplotlib correctly separate and display the different land cover categories.
>bounds = values_list + [7]

This line creates a normalisation rule using BoundaryNorm(). The function maps the raster class values to the correct colours in the custom colour map. The variable bounds defines the class intervals, while c_map.N specifies the total number of colours available in the colour map.
>norm = colors.BoundaryNorm(bounds, c_map.N)

This line displays the classified raster image using the imshow() plotting function. The raster dataset stored in predicted_int is visualised using the custom colour map c_map and the class boundary normalisation norm.
>predicted_int.plot.imshow(cmap=c_map, norm=norm, size=10)

This line displays all unique values contained within the classified raster dataset stored in predicted_int. The np.unique() function from NumPy scans through the raster and identifies every distinct class value that appears in the prediction results.
>print(np.unique(predicted_int))

Prints the contents of the variable values_list to the notebook output.
>print(values_list)

This line exports the classified raster dataset stored in predicted_int as a Cloud Optimized GeoTIFF (COG) file using the write_cog() function from the Open Data Cube (odc) library.
>predicted_int.odc.write_cog(f"{version}-prediction-4.tiff", Overwrite=True)